- apo(automatic prompt optimization)
    - 适用场景
        - 稳定长期复用的 prompts
        - 一定数量的跟真实业务对齐的 training dataset（切出部分的 validation dataset）
    - 算法
        - MIPRO-v2
        - GEPA
- references
    - https://huggingface.co/blog/dleemiller/auto-prompt-opt-dspy-cross-encoders

### MIPRO-v2

- STEP 1: BOOTSTRAP FEWSHOT EXAMPLES
    - These will be used as few-shot example candidates for our program and for creating instructions.
- STEP 2: PROPOSE INSTRUCTION CANDIDATES
    -  We will use the few-shot examples from the previous step, a generated dataset summary, a summary of the program code, and a randomly selected prompting tip to propose instructions.
-  STEP 3: FINDING OPTIMAL PROMPT PARAMETERS

### demo

- https://dspy.ai/learn/optimization/optimizers/
- hotpotqa
    - https://huggingface.co/datasets/hotpotqa/hotpot_qa
- a dspy.ColBERTv2 server to retrieve top matches from Wikipedia and parse them

In [2]:
import dspy
from dspy.datasets import HotPotQA
from dotenv import load_dotenv, find_dotenv

In [3]:
assert load_dotenv(find_dotenv(), override=True)

In [10]:
dspy.configure(lm=dspy.LM('openai/gpt-4.1-nano', ))

In [14]:
for x in HotPotQA(train_seed=2024, train_size=500).train:
    print(x)
    break

Example({'question': 'Are Smyrnium and Nymania both types of plant?', 'answer': 'yes'}) (input_keys=None)


In [16]:
def search(query: str) -> list[str]:
    """Retrieves abstracts from Wikipedia."""
    results = dspy.ColBERTv2(url='http://20.102.90.50:2017/wiki17_abstracts')(query, k=3)
    return [x['text'] for x in results]

trainset = [x.with_inputs('question') for x in HotPotQA(train_seed=2024, train_size=100).train]
react = dspy.ReAct("question -> answer", tools=[search])

tp = dspy.MIPROv2(metric=dspy.evaluate.answer_exact_match, auto="light", num_threads=24)
optimized_react = tp.compile(react, trainset=trainset)

2025/12/05 09:59:36 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 20
minibatch: True
num_fewshot_candidates: 6
num_instruct_candidates: 3
valset size: 80

2025/12/05 09:59:36 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2025/12/05 09:59:36 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2025/12/05 09:59:36 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=6 sets of demonstrations...


Bootstrapping set 1/6
Bootstrapping set 2/6
Bootstrapping set 3/6


 95%|██████████████████████████████████████████████████████████████████████████████████████▍    | 19/20 [02:03<00:06,  6.48s/it]


Bootstrapped 4 full traces after 19 examples for up to 1 rounds, amounting to 19 attempts.
Bootstrapping set 4/6


 50%|█████████████████████████████████████████████▌                                             | 10/20 [00:28<00:28,  2.84s/it]


...

2025/12/05 10:06:46 WARNING dspy.utils.parallelizer: Execution cancelled due to errors or interruption.
2025/12/05 10:06:46 ERROR dspy.teleprompt.utils: An exception occurred during evaluation
Traceback (most recent call last):
  File "/home/zhangchunhui/miniconda3/envs/verl/lib/python3.12/site-packages/dspy/teleprompt/utils.py", line 56, in eval_candidate_program
    return evaluate(
           ^^^^^^^^^
  File "/home/zhangchunhui/miniconda3/envs/verl/lib/python3.12/site-packages/dspy/utils/callback.py", line 326, in sync_wrapper
    return fn(instance, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/zhangchunhui/miniconda3/envs/verl/lib/python3.12/site-packages/dspy/evaluate/evaluate.py", line 175, in __call__
    results = executor.execute(process_item, devset)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/zhangchunhui/miniconda3/envs/verl/lib/python3.12/site-packages/dspy/utils/parallelizer.py", line 50, in execute
    return self._execu


  0%|                                                                                                    | 0/35 [00:00<?, ?it/s]

2025/12/05 10:06:47 WARNING dspy.utils.parallelizer: SIGINT received. Cancelling.
[W 2025-12-05 10:06:47,133] Trial 11 failed with parameters: {'0_predictor_instruction': 1, '0_predictor_demos': 3, '1_predictor_instruction': 2, '1_predictor_demos': 3} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/zhangchunhui/miniconda3/envs/verl/lib/python3.12/site-packages/optuna/study/_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/home/zhangchunhui/miniconda3/envs/verl/lib/python3.12/site-packages/dspy/teleprompt/mipro_optimizer_v2.py", line 510, in objective
    score = eval_candidate_program(batch_size, valset, candidate_program, evaluate, self.rng).score
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/zhangchunhui/miniconda3/envs/verl/lib/python3.12/site-packages/dspy/teleprompt/utils.py", line 56, in eval_candidat

KeyboardInterrupt: 

2025/12/05 10:06:47 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2025/12/05 10:06:47 ERROR dspy.utils.parallelizer: Error for Example({'question': 'What profession did both P. G. Wodehouse and Mordecai Richler take up?', 'answer': 'author'}) (input_keys={'question'}): litellm.RateLimitError: RateLimitError: OpenAIException - Rate limit reached for gpt-4.1-nano in organization org-N5qDVJCUIa4YWZtXEr9L8yj4 on tokens per min (TPM): Limit 200000, Used 197949, Requested 6173. Please try again in 1.236s. Visit https://platform.openai.com/account/rate-limits to learn more.. Set `provide_traceback=True` for traceback.
2025/12/05 10:06:47 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2025/12/05 10:06:48 ERROR dspy.utils.parallelizer: Error for Example({'question': 'What team does the athlete featured on the cover of the FIFA 17 video game play for?', 'answer': 'Borussia Dortmund'})